# HW3 - Transformers and Language Modeling
In this homework you will implement a Decoder Transformer model. You will then use these as the basis to train a (small) Language Model from scratch.

## Google Colab [Optional]

If you are using Google Colab, configure and run the cell below to mount your Google Drive and set up the correct working directory for the assignment. You may ignore this cell if you are working locally.

In [ ]:
from google.colab import drive
import os

# 1. Mounting Google Drive: This allows Colab to access files in your Google Drive
drive.mount('/content/drive')

# 2. Tell Colab where to find your assignment files and where to save your work

# TODO: Enter the relative path in your Google Drive of the assignment.
FOLDERNAME = "Deep Learning Homework/hw3_code"

assert FOLDERNAME is not None, "[!] Enter the foldername."

FULL_PATH = "/content/drive/MyDrive/" + FOLDERNAME
assert os.path.exists(FULL_PATH), "Make sure your FOLDERNAME is correct"

In [ ]:
%cd $FULL_PATH

In [ ]:
!uv pip install -r requirements.txt --system

## Embeddings

Recall from lecture that unlike a RNN, a Transformer does not inherently capture positional information in the forward pass. Because of this, we need to add a signal which encodes the position of each token in its embedding.

Your first task is to implement the embedding lookup, including the addition of positional encodings. We have already provided the necessary parameters inside of `Embedding`.

```python
self.vocab_embeddings = nn.Embedding(config.vocab_size, config.n_embd)
self.position_embeddings = nn.Embedding(config.block_size, config.n_embd)
```

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_embedding

## Multi-head Self-Attention
Attention can be computed in matrix-form using the following formula:

$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

We want to have multiple self-attention operations. Each of these is called a head with each head applied to some portion of the input.

$head_i = Attention(W_Q^i X, W_K^i X, W_V^i X)$

Here, we'll use GPT-style Multi-headed Self-Attention which fragments the head into pieces and applies one head to each fragment. The fragments are then concatenated together to reconstruct the transformed input and projected with a feed-forward layer.

$MultiHead(Q, K, V) = Concat(head_1, ..., head_h)W^O$

Note that while this is "Multi-head", all heads can be computed in parallel with a single matrix multiplication. You can find an in-depth description of this in the reference linked in the code.


We provide the needed weights in `GenericSelfAttention.__init__`

```python
# Note that we need this to be true in GPT-style MHA
# Knowing this might come in handy :)
assert config.n_embd % config.n_head == 0

# Note: These could be a single batched linear layer
# but we separate them for simplicity of implementation.
self.k = nn.Linear(config.n_embd, config.n_embd)
self.v = nn.Linear(config.n_embd, config.n_embd)
self.q = nn.Linear(config.n_embd, config.n_embd)
# output projection
self.c_proj = nn.Linear(config.n_embd, config.n_embd)
# regularization
self.attn_dropout = nn.Dropout(config.attn_pdrop)
self.hidden_dropout = nn.Dropout(config.hidden_pdrop)

self.n_head = config.n_head
self.n_embd = config.n_embd
```

Note: a few common mistakes in Multi-Headed Attention may cause implementations to fail test_transformer_hidden by < 1 and > 1e-4. If you are stuck, here are some hints:

1)  `reshape` and `transpose` are not equivalent functions. You'll need to use both of these functions in a correct implementation of MHA. See the example below to illustrate how these methods differ:


```python
a = torch.arange(0,3).repeat(4).reshape(1,3,4)
print("Swapping Last Dimension Using Transpose")
print(a.transpose(1, 2))
print()
print("Reshaping Tensor (doesn't actually Transpose)")
print(a.reshape(1, 4, 3))
```

2) The tests require assumptions about where dropout is applied, according to what GPT-2 does in practice. Since this is a somewhat arbitrary decision in the original GPT-2 implementation, we are very specific about this in the docstrings for `GenericSelfAttention.forward`.

3) The implementation is Multi-Headed attention. You must isolate the heads as an additional dimension from the original feature dimenstion e.g. (batch_size, sequence_length, embedding_dimension) -> (batch_size, sequence_length, head_num, embedding_dimension // head_num) -> (batch_size, head_num, sequence_length, embedding_dimension // head_num); making head as a new batch dimenstion will make your calculation fully vectorized, see (4) for more hints.

4) Your implementation should not use a for loop over n_heads. There's a for-loop version that is theoretically equivalent, but it will make training/inference very slow. The instructions specify that you should compute all heads in parallel and the tests are designed to capture the Floating Point drift introduced by the for-loop version.

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_mha

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_mha_mask

## Putting it all together

Using our Embedding Layer, the above Transformer Block using our Multi-head attention, and a simple classification head we have all the pieces we need for a Transformer language model.

For the forward pass, you'll want to
1) first embed our inputs,
2) apply each transformer layer sequentially,
3) apply layer normalization after the final transformer block and before the language modeling head
4) get logits for each possible output word using a classification layer (often called a language modeling head).

All the parameters you'll need come from `GenericTransformer` and the code blocks above your code section.

```python
self.transformer = nn.ModuleDict(
    dict(
        embedding=Embedding(config),
        h=nn.ModuleList(
            [block(config) for _ in range(config.n_layer)]
        ),
        ln_f=nn.LayerNorm(config.n_embd),
    )
)
self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
```

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_transformer

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_transformer_block

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_transformer_loss

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_transformer_hidden

## Implement an Encoder Transformer

Encoders, like the BERT model you learned about in lecture, utilize bi-directional attention. This means that in the sequence "A B C", the representation for token "B" will be influenced by tokens A *and* C. When all tokens can attend to all other tokens, the attention_mask is just a matrix of ones.

However, since sentences come in a wide range of lengths, we need a way to batch sequences of different lengths together in order to maximize our GPU throughput. The most common way of doing this is called "Padding". When you pad an input, you add additional "pad" tokens to make it the same length as the longest sequence in a batch. For example, if we wanted to batch "A B C" and "A B C D" together, we would add a "pad" token to "A B C". Our resulting batch would be \["A B C \<pad\>", "A B C D"\].

Since these pad tokens are meaningless, we do not want the real tokens to pay attention to the pad tokens. Below, you'll write a function to create such an attention mask for padded sequences given a tensor which contains the number of valid leading tokens for each batch.

Clarification: In practice, we train on sequences of varying lengths, so we will sometimes need extra pad tokens to make all the sequences the same lengths. We don't backpropagate through the extra pad tokens by setting ignore_index to -1 in F.cross_entropy. To make our implementation simple, set the pad token's attention mask to be the same as the last non-pad token's attention mask.

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_encoder

## Implement an Decoder Transformer

Unlike Encoders, Decoders are a "causal" model, meaning that each prediction is only influenced by the tokens which came earlier than it in the input. While "Encoders" and "Decoders" are often discussed as different types of models, the only core difference is the attention mask used.

For decoders, we want the attention mask for each token to only include the previous tokens in the sequence. Despite being functionally very different models, a "Decoder" can be implemented with just a one line change of the "Encoder" attention mask. You'll implement this below.

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_decoder

In [ ]:
!python -m pytest ./tests/test_transformer.py::test_generate

In [ ]:
!python -m pytest ./tests/test_train.py

# You've implemented a language model!


## Training a Language Model from Scratch

In `train.py`, we set up code which loads a training dataset for pure Decoder models. We want to train a model to translate between English and Pig-Latin!

Below is an implementation which achieves between 40 and 50 percent accuracy. Modify the tokenizer in `src/data.py`, architecture in `src/transformer.py`, or hyperparameters in the arguments below to decrease the loss as much as possible and drive Exact Match above 80%.

- Training might take a while (>10 minutes), so be patient.
- You may run into issues with mismatched devices and shapes here. This is a normal part of working with PyTorch. It is quite difficult to write tests that cover the full range of possible issues, treat this as a chance to practice debugging. Feel free to ask conceptual questions on Piazza or get specific help in office hours, but do not just directly post your error and code on Piazza.

Run the cells in the Writeup below to evaluate your model. You can make new cells as you please, but just make sure to clear any output cells that are not a part of your Writeup submission.


### Baseline

In [ ]:
!python train.py \
  --learning_rate 5e-4 \
  --batch_size 16 \
  --max_iters 10000 \
  --dec_n_layer 4 \
  --dec_n_embd 64 \
  --dec_n_head 2 \
  --log_interval 100 \
  --checkpoint_dir checkpoints

### Hyperparameter Tuning

In [ ]:
# feel free to change the hyperparameters to see what works best
!python train.py \
  --learning_rate 3e-4 \
  --batch_size 128 \
  --max_iters 10000 \
  --dec_n_layer 6 \
  --dec_n_embd 256 \
  --dec_n_head 8 \
  --log_interval 100 \
  --checkpoint_dir checkpoints_tuned

# **Homework 3 Writeup**

- Name: `Venkata Aashutosh Aripirala`
- GT Email: `av84@gatech.edu`
- GT ID: `904173725`



Report what you changed and your intuitions for why you changed it the Writeup below. For submission, make sure to delete all output cells above for tests/training, but leave the cells below from evaluation and assign them to corresponding questions on Gradescope.

## Results[2 pts]  
#### Before Hyperparameter Tuning

BLEU: `89.6142`

Exact Match: `66.00%`

In [18]:
!python evaluate.py checkpoints/checkpoint_iter_10000.pt

2026-10-18 00:38:33,449 - INFO - Starting transformer evaluation script
2026-10-18 00:38:33,449 - INFO - Checkpoint: checkpoints/checkpoint_iter_10000.pt
2026-10-18 00:38:33,449 - INFO - Temperature: 0.1
2026-10-18 00:38:33,449 - INFO - Using device: cuda
2026-10-18 00:38:33,449 - INFO - Loading checkpoint from checkpoints/checkpoint_iter_10000.pt
2026-10-18 00:38:33,736 - INFO - Loaded checkpoint from iteration 10000 with loss 0.754585862159729
2026-10-18 00:38:33,736 - INFO - Loading piglatin dataset split: validation...
Map: 100% 1000/1000 [00:00<00:00, 23546.84 examples/s]
2026-10-18 00:38:34,692 - INFO - Loaded 1000 samples from validation split
number of parameters: 0.23M
2026-10-18 00:38:34,701 - INFO - Created Decoder model with 4 layers, 64 embedding dim, 2 heads
2026-10-18 00:38:34,707 - INFO - Model loaded with 245,888 parameters
2026-10-18 00:38:34,707 - INFO - Generating 5 sample translations...
2026-10-18 00:38:35,475 - INFO - Sample 1: translate this to piglatin -> ansla

#### After Hyperparameter Tuning

BLEU: `97.7748`

Exact Match: `91.90%`

In [19]:
!python evaluate.py checkpoints_tuned/checkpoint_iter_10000.pt

2026-10-18 00:47:35,877 - INFO - Starting transformer evaluation script
2026-10-18 00:47:35,877 - INFO - Checkpoint: checkpoints_tuned/checkpoint_iter_10000.pt
2026-10-18 00:47:35,877 - INFO - Temperature: 0.1
2026-10-18 00:47:35,878 - INFO - Using device: cuda
2026-10-18 00:47:35,878 - INFO - Loading checkpoint from checkpoints_tuned/checkpoint_iter_10000.pt
2026-10-18 00:47:36,192 - INFO - Loaded checkpoint from iteration 10000 with loss 0.23900318145751953
2026-10-18 00:47:36,192 - INFO - Loading piglatin dataset split: validation...
2026-10-18 00:47:37,073 - INFO - Loaded 1000 samples from validation split
number of parameters: 4.86M
2026-10-18 00:47:37,160 - INFO - Created Decoder model with 6 layers, 256 embedding dim, 8 heads
2026-10-18 00:47:37,181 - INFO - Model loaded with 4,922,368 parameters
2026-10-18 00:47:37,181 - INFO - Generating 5 sample translations...
2026-10-18 00:47:38,134 - INFO - Sample 1: translate this to piglatin -> anslate-tray is-thay o-tay iglatin-pay
2026

## Analysis[2 pts]  


What did you change to improve your models’ performance? Why did you make these specific changes
```
Increasing Model Capacity: Translation requires a correct identification of word structure, and perform complex reordering of syllables. A tiny network with only 245k parameters lacks the capacity to map robust structural representations. I expanded to 6 layers, 256 embedding dimensions, and 8 attention heads bringing the parameter count to ~4.9M, giving the model strong representational powers that it needs.

Increased Batch Size: Using a large batch size (I increased it from 16 to 128) to allow for smoother and more accurate gradient estimates, stabilizing training dynamics and accelerating convergenece as well.

While I also changed learning rate a bit, I don't think that was a major driver towards performance improvement especially considering the changes to attention heads and embedding dimensions.
```

Were the results what you expected? Why or why not?

`The results were along highly expected lines. Scaling the model capacity in these types of tasks I always thought was bound to increase the performance. What also happened was a smoother gradient due to the batch size, stabilizing the training better. `



# Submission
Colab may take some time to update on the cloud file system, so if you find that running this cell does not generate the latest version of your notebook, **make sure to save the notebook** and wait a few minutes before trying again, or just download the notebook and run the command locally.

In [ ]:
# on colab, you may need to install chromium to convert the notebook to pdf
!playwright install-deps chromium
!playwright install chromium

In [22]:
!python ./collect_submission.py

=== HW3 Submission Collection Script ===

Converting notebook to PDF...
Successfully converted main.ipynb to hw3_notebook_submission.pdf

Creating zip archive...
Checking required paths...
All required paths found
  Added file: src/transformer.py
Created hw3_code_submission.zip
Zip file verified (size: 5190 bytes)

Performing final verification...
PDF verified: hw3_notebook_submission.pdf (117149 bytes)
ZIP verified: hw3_code_submission.zip (5190 bytes)

=== Submission Collection Complete ===
Generated: hw3_notebook_submission.pdf
Generated: hw3_code_submission.zip

Please submit these files to Gradescope:
  - hw3_notebook_submission.pdf
  - hw3_code_submission.zip

Congratulations on completing the assignment!
